# 04 - Pricing Scenario Logic Deep Dive

**Dynamic Pricing Engine** — assumption-based decision support

The production pricing layer intentionally does **not** estimate a causal price
elasticity from observational Airbnb calendar data.

Instead, it asks:

> If the demand proxy responded to price with a stated sensitivity, which
> candidate price would maximize the corresponding revenue proxy?

The scenario response is:

\[
D(P) = D_0 \exp\left[-s\left(\frac{P}{P_0} - 1\right)\right]
\]

where:

- \(D_0\) = model-derived demand/unavailability proxy at the reference price
- \(P_0\) = reference price
- \(s\) = **assumed** sensitivity, not an estimated causal elasticity
- \(D(P)\) = scenario demand proxy

The decision score is:

\[
R_{\text{proxy}}(P) = P \times D(P)
\]

This is a scenario-analysis score, **not a claim of realized revenue**.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

from src.models.scenario_engine import (
    SENSITIVITY_PRESETS,
    compute_scenario_curve,
    evaluate_sensitivity_presets,
    optimize_price_scenario,
    scenario_demand_proxy,
)
from src.utils.config import load_config

config = load_config()

## 1. Sensitivity assumptions

In [ ]:
base_demand_proxy = 0.70
reference_price = 150.0

rows = [
    {"scenario": name, "assumed_sensitivity": value}
    for name, value in SENSITIVITY_PRESETS.items()
]
pd.DataFrame(rows)

In [ ]:
candidate_prices = [75, 100, 125, 150, 175, 200, 225]

table = []
for name, sensitivity in SENSITIVITY_PRESETS.items():
    for price in candidate_prices:
        table.append(
            {
                "scenario": name,
                "price": price,
                "demand_proxy": scenario_demand_proxy(
                    price=price,
                    base_demand_proxy=base_demand_proxy,
                    reference_price=reference_price,
                    sensitivity=sensitivity,
                ),
            }
        )

pd.DataFrame(table).head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for name, sensitivity in SENSITIVITY_PRESETS.items():
    curve = compute_scenario_curve(
        base_demand_proxy=base_demand_proxy,
        reference_price=reference_price,
        sensitivity=sensitivity,
        floor_price=75,
        ceiling_price=225,
        config=config,
    )
    ax.plot(
        curve["prices"],
        curve["demand_proxies"],
        label=f"{name} ({sensitivity:.2f})",
    )

ax.axvline(reference_price, linestyle="--", label="reference price")
ax.set_title("Demand-Proxy Curves Under Explicit Sensitivity Assumptions")
ax.set_xlabel("Candidate price")
ax.set_ylabel("Demand / calendar-unavailability proxy")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Revenue-proxy surfaces

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

results = evaluate_sensitivity_presets(
    base_demand_proxy=base_demand_proxy,
    reference_price=reference_price,
    floor_price=75,
    ceiling_price=225,
    config=config,
)

for name, sensitivity in SENSITIVITY_PRESETS.items():
    curve = compute_scenario_curve(
        base_demand_proxy=base_demand_proxy,
        reference_price=reference_price,
        sensitivity=sensitivity,
        floor_price=75,
        ceiling_price=225,
        config=config,
    )
    result = results[name]

    ax.plot(
        curve["prices"],
        curve["revenue_proxies"],
        label=f"{name} ({sensitivity:.2f})",
    )
    ax.scatter(
        [result.recommended_price],
        [result.revenue_proxy_at_recommended],
    )

ax.axvline(reference_price, linestyle="--", label="reference price")
ax.set_title("Revenue-Proxy Curves and Scenario Recommendations")
ax.set_xlabel("Candidate price")
ax.set_ylabel("Revenue proxy")
ax.legend()
plt.tight_layout()
plt.show()

pd.DataFrame(
    [
        {
            "scenario": name,
            "sensitivity": result.sensitivity,
            "recommended_price": result.recommended_price,
            "demand_proxy_at_recommended": result.demand_proxy_at_recommended,
            "revenue_proxy_change_pct": result.revenue_proxy_change_pct,
        }
        for name, result in results.items()
    ]
)

## 3. Known-optimum validation

For

\[
R(P) = P D_0 \exp\left[-s\left(\frac{P}{P_0}-1\right)\right],
\]

the unconstrained first-order condition gives:

\[
P^* = \frac{P_0}{s}.
\]

That gives us a useful synthetic validation: when the optimum lies inside the
configured bounds, the numerical scenario engine should recover it.

In [ ]:
sensitivity = 1.25
reference_price = 150.0
analytical_optimum = reference_price / sensitivity

numerical = optimize_price_scenario(
    base_demand_proxy=0.50,
    reference_price=reference_price,
    sensitivity=sensitivity,
    scenario_name="synthetic_check",
    floor_price=75,
    ceiling_price=225,
    n_points=3001,
    config=config,
)

print(f"Assumed sensitivity: {sensitivity}")
print(f"Analytical optimum:  ${analytical_optimum:.2f}")
print(f"Numerical optimum:   ${numerical.recommended_price:.2f}")
print(
    "Match:",
    abs(analytical_optimum - numerical.recommended_price) < 0.20,
)

## 4. Bounds and business constraints

In [ ]:
opt_cfg = config["optimization"]

print("Configured price bounds")
print(
    "  floor multiplier:",
    opt_cfg["price_floor_multiplier"],
)
print(
    "  ceiling multiplier:",
    opt_cfg["price_ceiling_multiplier"],
)

bounded = optimize_price_scenario(
    base_demand_proxy=0.70,
    reference_price=150,
    sensitivity=SENSITIVITY_PRESETS["low"],
    config=config,
)

print("\nLow-sensitivity bounded scenario")
print(bounded)

## 5. Scenario comparison across listing profiles

In [ ]:
profiles = [
    {"name": "Budget private room", "price": 60, "demand_proxy": 0.80},
    {"name": "Mid-range apartment", "price": 150, "demand_proxy": 0.65},
    {"name": "Luxury penthouse", "price": 400, "demand_proxy": 0.40},
    {"name": "Holiday hot spot", "price": 200, "demand_proxy": 0.90},
    {"name": "Low-demand area", "price": 100, "demand_proxy": 0.30},
]

rows = []
for profile in profiles:
    scenario_results = evaluate_sensitivity_presets(
        base_demand_proxy=profile["demand_proxy"],
        reference_price=profile["price"],
        config=config,
    )

    for scenario_name, result in scenario_results.items():
        rows.append(
            {
                "profile": profile["name"],
                "scenario": scenario_name,
                "reference_price": profile["price"],
                "recommended_price": result.recommended_price,
                "revenue_proxy_change_pct": result.revenue_proxy_change_pct,
            }
        )

pd.DataFrame(rows)

## 6. Interpretation and limitations

The scenario engine is deliberately narrower than a causal pricing model.

**What it can do**
- expose the recommendation's sensitivity to alternative price-response assumptions;
- enforce configured price bounds;
- generate reproducible price/demand-proxy/revenue-proxy curves;
- recover known optima on synthetic response functions.

**What it does not claim**
- that the Airbnb calendar data identify causal price elasticity;
- that changing price will cause the predicted change in demand;
- that the revenue proxy equals realized revenue;
- that one sensitivity preset is universally correct.

A production causal-pricing system would require stronger identification, such
as randomized price experiments, valid instruments, or another defensible
causal design.